In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
import bs4
import requests
from langchain_core.documents import Document
from dotenv import load_dotenv
from langchain_text_splitters import MarkdownHeaderTextSplitter,RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()
DB_PATH = "../RAG/chroma_langchain_db"

C:\Users\DHIRAJ\AppData\Local\Temp\ipykernel_25388\229619480.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [2]:
# Remove the current db

import shutil

shutil.rmtree(DB_PATH)

## Initialize Tools

In [3]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

# embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")


embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)


vector_store = Chroma(
    collection_name="portfolio",
    embedding_function=embeddings,
    persist_directory=DB_PATH
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Loading Docs

In [4]:
loader = DirectoryLoader(
    path="../context_documents",
    glob="**/*.md",
    loader_cls=TextLoader
)

documents = loader.load()

for doc in documents:
    filepath = Path(doc.metadata["source"])

    doc.metadata.update({
        "filename": filepath.name,
        "category": filepath.stem,
        "filetype": "markdown"
    })

print(documents[0].metadata)

print(f"Total characters in doc 1: {len(documents[0].page_content)}")

{'source': '..\\context_documents\\01_identity_and_background.md', 'filename': '01_identity_and_background.md', 'category': '01_identity_and_background', 'filetype': 'markdown'}
Total characters in doc 1: 7234


## Chunking

In [5]:
headers_to_split_on = [
    ("#", "section")
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

md_header_splits = []

for doc in documents:
    splits = markdown_splitter.split_text(doc.page_content)

    for split in splits:
        split.metadata.update(doc.metadata)
        md_header_splits.append(split)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    add_start_index=True
)

final_chunks = text_splitter.split_documents(md_header_splits)

print(f"Total chunks: {len(final_chunks)}")

Total chunks: 35


In [7]:
for idx, chunk in enumerate(final_chunks):

    chunk.metadata["citation_id"] = f"C{idx:04d}"
    filename = chunk.metadata.get("filename", "").lower()

    if "personality" in filename:
        chunk.metadata["type"] = "personality"

    elif "skills" in filename or "projects" in filename:
        chunk.metadata["type"] = "technical"

    elif "faq" in filename:
        chunk.metadata["type"] = "recruiter"

    elif "opinions" in filename:
        chunk.metadata["type"] = "beliefs"

    else:
        chunk.metadata["type"] = "general"

## Saving Vector chunks in vector_store(ChromaDB)

In [8]:
# document_ids = vector_store.add_documents(documents=final_chunks)
# print(document_ids[:3])
import time
batch_size = 50

for i in range(0, len(final_chunks), batch_size):
    batch = final_chunks[i:i+batch_size]
    vector_store.add_documents(batch)
    print(f"Added batch {i//batch_size + 1}")


Added batch 1


## CrossEncoder Reranking using sentence_transformers

In [9]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-12-v2"
)

def rerank(query, docs, top_k=5):

    pairs = [
        (query, doc.page_content)
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        doc
        for doc, score in ranked_docs[:top_k]
    ]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## Hybrid retriever (BM25 + Vector Retriever)

In [ ]:
from langchain_community.retrievers import BM25Retriever

# BM25 retriever
bm25_retriever = BM25Retriever.from_documents(final_chunks)
bm25_retriever.k = 5

# Vector retriever
vector_retriever = vector_store.as_retriever(
    search_kwargs={"k": 5}
)

def hybrid_search(query, k=5):

    bm25_docs = bm25_retriever.invoke(query)
    vector_docs = vector_retriever.invoke(query)

    scores = {}

    for rank, doc in enumerate(bm25_docs):
        scores[doc.page_content] = scores.get(doc.page_content, 0) + 1/(rank+60)

    for rank, doc in enumerate(vector_docs):
        scores[doc.page_content] = scores.get(doc.page_content, 0) + 1/(rank+60)

    doc_map = {
        doc.page_content: doc
        for doc in bm25_docs + vector_docs
    }

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        doc_map[content]
        for content, _ in ranked[:k]
    ]

In [11]:
def retrieve(query):

    candidates = hybrid_search(
        query,
        k=10
    )

    final_docs = rerank(
        query,
        candidates,
        top_k=5
    )

    return final_docs

## RAG Query Optimization

In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0
)

query_optimizer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a query optimization assistant for a RAG system.

The RAG is used by chatbot which is mimicing a person Dhiraj.
All information are about Dhiraj only, not the AI

Your task:
- Rewrite the user's query to maximize retrieval quality.
- Preserve the original intent.
- Add relevant keywords, synonyms, and context if helpful.
- Expand abbreviations when appropriate.
- Make the query explicit and self-contained.
- Optimize for both BM25 keyword search and vector search.

Return ONLY the optimized query.
Do not explain your reasoning.
"""
    ),
    ("human", "{query}")
])

query_optimizer = query_optimizer_prompt | model

def optimize_query(query: str) -> str:
    response = query_optimizer.invoke(
        {"query": query}
    )

    return response.content.strip()





## For Testing : Unoptimized Query (Due to API rate limitations) 

In [13]:
query = "What do you do in your free time?"

print(query)
results = retrieve(
    query
)

for doc in results:
    print("\n" + "="*50)
    print(doc.page_content)

What do you do in your free time?

```yaml
id: hobbies
category: personal
tags:
- hobbies
- interests
```  
## Indoor Hobbies  
- Gaming
- Watching TV shows and Movies  
## Outdoor Hobbies  
- Playing Badminton and Swimming
- Walking  
## Creative Interests  
- Cooking  
## Technical Interests  
- New technology in Artificial Intelligence and Machine Learning
- Solving DSA problems  
## Question Variants  
- What do you do in your free time?
- What are your hobbies?
- What interests you outside academics?
- What do you enjoy doing for fun?  
---

---  
### Question: How do you learn new technologies?  
## Answer:  
Keywords:
learning, adaptability, self learning, curiosity  
---  
### Question: Where do you see yourself in 5 years?  
## Answer:  
Keywords:
future goals, career vision, aspirations  
---  
## Casual Questions  
### Question: Favorite food?  
## Answer:  
Keywords:
food, favorites, hobbies  
---  
### Question: Favorite music while coding?  
## Answer:  
Keywords:
music, 

## For Testing: Test Retrieval

In [ ]:
query = "Talk about your work Experiences and internships"

optimized_query = optimize_query(query)
print(optimized_query)
results = retrieve(
    optimized_query
)

for doc in results:
    print("\n" + "="*50)
    print(doc.page_content)

## For Testing: Chatbot

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
import re
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.3
)

def generate_answer(query):

    docs = retrieve(query)

    context = "\n\n".join(
            f"""
        [CITATION: {doc.metadata.get('citation_id')}]

        SOURCE:
        {doc.metadata.get('filename')}

        CONTENT:
        {doc.page_content}
        """
            for doc in docs
            )

    prompt = f"""
    You are Dhiraj.

    Answer naturally like Dhiraj would. Sound natural in a human tone 

    Use the retrieved context below to answer accurately.

    Do not mention retrieval systems, vector databases, or context documen

    Answer ONLY using the provided context.

    Every factual statement MUST include a citation.

    Use citations in this format:

    [C0001]

    Example:

    I enjoy working with Python and AI systems. [C0007]

    If multiple chunks support a statement:

    [C0007][C0012]

    Do not make any factual claim unless it is supported by at least one citation.

    If the answer is not present in the context,
    say that you do not have enough information.

    CONTEXT:
    {context}

    QUESTION:
    {query}

    ANSWER:
    """

    response = llm.invoke(
        [HumanMessage(content=prompt)]
    )

    answer = response.content

    citations = re.findall(r"\[C\d+\]", answer)

    if not citations:
        print("WARNING: No citations generated")

    return answer
    return response.content


def chatbot():

    print("=" * 60)
    print("Portfolio RAG Chatbot")
    print("Type 'exit' to quit")
    print("=" * 60)

    while True:

        query = input("\nYou: ")

        if query.lower() in ["exit", "quit"]:
            print("Goodbye!")
            break

        try:

            answer = generate_answer(query)

            print("\nBot:")
            print(answer)

        except Exception as e:

            print("\nError:")
            print(e)


chatbot()